In [8]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    groq_api_key=groq_api_key=os.getenv("GROQ_API_KEY"),
    temperature=0.0,
    # max_retries=2,
    # # other params...
)
response=llm.invoke("The First person to land on moon was ..")
print(response.content)

The first person to land on the moon was Neil Armstrong. He stepped out of the lunar module Eagle and onto the moon's surface on July 20, 1969, during the Apollo 11 mission. Armstrong famously declared, "That's one small step for man, one giant leap for mankind," as he became the first human to set foot on the moon.


In [11]:
from langchain_community.document_loaders import WebBaseLoader
#https://www.databricks.com/company/careers/open-positions?department=Engineering&location=Vancouver,%20Canada
loader = WebBaseLoader("https://www.databricks.com/company/careers/engineering---pipeline/senior-software-engineer---backend-8093295002?gh_jid=8093295002&gh_src=62a881d62")
page_data=loader.load().pop().page_content
print(page_data)

USER_AGENT environment variable not set, consider setting it to identify your requests.


Senior Software Engineer - Backend - DatabricksSkip to main contentLoginWhy Databricks DiscoverFor ExecutivesFor Startups Lakehouse Architecture Mosaic ResearchCustomersCustomer StoriesPartnersPartner OverviewExplore the Databricks partner ecosystem Partner SpotlightFeatured partner announcementsPartner ProgramExplore benefits, tiers and how to become a partnerCloud ProvidersDatabricks on AWS, Azure and GCPFind a PartnerDiscover Databricks partners for your needsPartner SolutionsFind custom industry and migration solutionsProduct Databricks PlatformPlatform OverviewA unified platform for data, analytics and AIData ManagementData reliability, security and performanceSharingAn open, secure, zero-copy sharing for all dataData WarehousingServerless data warehouse for SQL analyticsGovernanceUnified governance for all data, analytics and AI assetsData EngineeringETL and orchestration for batch and streaming dataArtificial IntelligenceBuild and deploy ML and GenAI applicationsData ScienceColl

In [22]:
from langchain_core.prompts import PromptTemplate
prompt_extract = PromptTemplate.from_template("""
### SCRAPED TEXT FROM WEBSITE:
{page_data}

### INSTRUCTION:
Extract the job postings and return them in JSON format with keys:
'role','experience','skills','description'.

### OUTPUT RULES (VERY IMPORTANT):
- Return ONLY beautify JSON
- Do NOT wrap the output in ```json or ``` 

Return JSON now:
""")

chain_extract=prompt_extract|llm
res=chain_extract.invoke(input={'page_data':page_data})
print(res.content)


{
  "role": "Senior Software Engineer - Backend",
  "experience": "5+ years of production level experience in either Java, Scala or C++",
  "skills": [
    "Java",
    "Scala",
    "C++",
    "algorithms and data structures",
    "large-scale distributed systems",
    "SaaS platform or Service-Oriented Architectures",
    "cloud technologies (e.g. AWS, Azure, GCP, Docker, or Kubernetes)",
    "security and systems that handle sensitive data",
    "SQL"
  ],
  "description": "Databricks is on a mission to simplify and democratize data and AI. We are looking for a Senior Software Engineer - Backend to join our team in Vancouver, Canada. The successful candidate will have experience developing large-scale distributed systems, working on a SaaS platform or with Service-Oriented Architectures, and have a strong foundation in algorithms and data structures."
}


In [25]:
from langchain_core.output_parsers import JsonOutputParser
json_parser=JsonOutputParser()
json_res=json_parser.parse(res.content)
json_res

{'role': 'Senior Software Engineer - Backend',
 'experience': '5+ years of production level experience in either Java, Scala or C++',
 'skills': ['Java',
  'Scala',
  'C++',
  'algorithms and data structures',
  'large-scale distributed systems',
  'SaaS platform or Service-Oriented Architectures',
  'cloud technologies (e.g. AWS, Azure, GCP, Docker, or Kubernetes)',
  'security and systems that handle sensitive data',
  'SQL'],
 'description': 'Databricks is on a mission to simplify and democratize data and AI. We are looking for a Senior Software Engineer - Backend to join our team in Vancouver, Canada. The successful candidate will have experience developing large-scale distributed systems, working on a SaaS platform or with Service-Oriented Architectures, and have a strong foundation in algorithms and data structures.'}

In [28]:
import pandas as pd
df=pd.read_csv("my_portfolio.csv")
df

,Techstack,Links
0,"React, Node.js, MongoDB",https://example.com/react-portfolio
1,"Angular,.NET, SQL Server",https://example.com/angular-portfolio
2,"Vue.js, Ruby on Rails, PostgreSQL",https://example.com/vue-portfolio
3,"Python, Django, MySQL",https://example.com/python-portfolio
4,"Java, Spring Boot, Oracle",https://example.com/java-portfolio
5,"Flutter, Firebase, GraphQL",https://example.com/flutter-portfolio
6,"WordPress, PHP, MySQL",https://example.com/wordpress-portfolio
7,"Magento, PHP, MySQL",https://example.com/magento-portfolio
8,"React Native, Node.js, MongoDB",https://example.com/react-native-portfolio
9,"iOS, Swift, Core Data",https://example.com/ios-portfolio


In [32]:
import chromadb
import uuid
chroma_client = chromadb.PersistentClient('coldemailvectorstore')
collection = chroma_client.get_or_create_collection(name="portfolio")
if not collection.count():
    for _,row in df.iterrows():
        collection.add(documents=row["Techstack"],
                        metadatas={"links": row["Links"]},
                        ids=[str(uuid.uuid4())])


C:\Users\PC\.cache\chroma\onnx_models\all-MiniLM-L6-v2\onnx.tar.gz: 100%|████████| 79.3M/79.3M [00:23<00:00, 3.48MiB/s]


In [36]:
job=json_res
job['skills']

['Java',
 'Scala',
 'C++',
 'algorithms and data structures',
 'large-scale distributed systems',
 'SaaS platform or Service-Oriented Architectures',
 'cloud technologies (e.g. AWS, Azure, GCP, Docker, or Kubernetes)',
 'security and systems that handle sensitive data',
 'SQL']

In [38]:
links= collection.query(query_texts=job['skills'],n_results=2).get('metadatas',[])
links

[[{'links': 'https://example.com/java-portfolio'},
  {'links': 'https://example.com/android-portfolio'}],
 [{'links': 'https://example.com/ml-python-portfolio'},
  {'links': 'https://example.com/kotlin-android-portfolio'}],
 [{'links': 'https://example.com/ml-python-portfolio'},
  {'links': 'https://example.com/magento-portfolio'}],
 [{'links': 'https://example.com/ml-python-portfolio'},
  {'links': 'https://example.com/magento-portfolio'}],
 [{'links': 'https://example.com/android-portfolio'},
  {'links': 'https://example.com/xamarin-portfolio'}],
 [{'links': 'https://example.com/xamarin-portfolio'},
  {'links': 'https://example.com/angular-portfolio'}],
 [{'links': 'https://example.com/xamarin-portfolio'},
  {'links': 'https://example.com/devops-portfolio'}],
 [{'links': 'https://example.com/magento-portfolio'},
  {'links': 'https://example.com/ios-portfolio'}],
 [{'links': 'https://example.com/magento-portfolio'},
  {'links': 'https://example.com/wordpress-portfolio'}]]

In [40]:
prompt_email = PromptTemplate.from_template(
        """
        ### JOB DESCRIPTION:
        {job_description}
        
        ### INSTRUCTION:
        You are Ashima, a business development executive at APK Consulting. APK is an AI & Software Consulting company dedicated to facilitating
        the seamless integration of business processes through automated tools. 
        Over our experience, we have empowered numerous enterprises with tailored solutions, fostering scalability, 
        process optimization, cost reduction, and heightened overall efficiency. 
        Your job is to write a cold email to the client regarding the job mentioned above describing the capability of AtliQ 
        in fulfilling their needs.
        Also add the most relevant ones from the following links to showcase APK's portfolio: {link_list}
        Remember you are Ashima, BDE at APK. 
        Do not provide a preamble.
        ### EMAIL (NO PREAMBLE):
        
        """
        )

chain_email = prompt_email | llm
res = chain_email.invoke({"job_description": str(job), "link_list": links})
print(res.content)

Subject: Expert Backend Solutions for Databricks' Senior Software Engineer Role

Dear Hiring Manager,

I came across the job posting for a Senior Software Engineer - Backend at Databricks, and I'm excited to introduce you to APK Consulting, a leading AI & Software Consulting company. With our expertise in developing large-scale distributed systems, SaaS platforms, and Service-Oriented Architectures, I believe we can provide the perfect fit for your requirements.

Our team at APK has extensive experience in working with Java, Scala, and C++, and we've successfully implemented algorithms and data structures, cloud technologies (AWS, Azure, GCP, Docker, Kubernetes), and security measures for systems handling sensitive data. We're confident that our skills align with your needs, and we'd love to discuss how we can contribute to Databricks' mission to simplify and democratize data and AI.

To give you a glimpse into our capabilities, I'd like to share some relevant portfolio links:
- https: